In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "redacted"
os.environ["HF_TOKEN"] = "redacted"
os.environ["WEAVIATE_URL"] = "redacted"
os.environ["WEAVIATE_API_KEY"] = "redacted"

In [ ]:
import numpy as np
np.set_printoptions(suppress=True, precision=10)

In [ ]:
from google.colab import userdata
userdata.get('HF_TOKEN')

In [ ]:
# example use of HF transformers library to predict sentiment of a prompt (comment made by student)

from transformers import pipeline

pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print(pipe("SEC545 class is awesome! I love it."))

Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.9998792409896851}]


In [ ]:
# generate tokens based on input prompt string
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-2")

# Input string
text = "Hello, I am learning how to tokenize text using tiktoken."

# Encode string into tokens
tokens = encoding.encode(text)

# Print each token and its decoded value
print("Tokens and their text:")
for token in tokens:
    print(f"{token} -> {encoding.decode([token])}")

# Print number of tokens
print("\nTotal tokens:", len(tokens))

Tokens and their text:
15496 -> Hello
11 -> ,
314 ->  I
716 ->  am
4673 ->  learning
703 ->  how
284 ->  to
11241 ->  token
1096 -> ize
2420 ->  text
1262 ->  using
256 ->  t
1134 -> ik
30001 -> token
13 -> .

Total tokens: 15


In [ ]:
# Obtain embeddings of the input prompt string
from openai import OpenAI

client = OpenAI()

# Input string
text = "Hello, I am learning how to compute embeddings."

model = "text-embedding-3-small"

# Compute embedding
response = client.embeddings.create(
    model=model,
    input=text
)

# Extract embedding vector
embedding = response.data[0].embedding

# Print results
print("Embedding length:", len(embedding))
print("First 10 values:", embedding[:5])


Embedding length: 1536
First 10 values: [-0.0022012002300471067, -0.043641701340675354, 0.019138826057314873, 0.006387450732290745, -0.0104575389996171]


In [ ]:
# Code to reverse embeddings to approximate clear text

from openai import OpenAI
import numpy as np

client = OpenAI()

# Suppose you already have your target embedding
target_embedding = embedding  # your stored vector

# A small reference corpus
texts = [
    "The cat sat on the mat.",
    "Embeddings convert text into numbers for semantic understanding.",
    "OpenAI provides text embedding models.",
    "Hello, I am learning how to compute."
]

# Compute embeddings for the reference corpus
embeddings = [
    client.embeddings.create(model="text-embedding-3-small", input=t).data[0].embedding
    for t in texts
]

# Compute cosine similarity
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Find most similar text
similarities = [cosine_similarity(target_embedding, e) for e in embeddings]
best_match_index = int(np.argmax(similarities))
print("Most similar text:", texts[best_match_index])
print("Similarity score:", similarities[best_match_index])


Most similar text: Hello, I am learning how to compute.
Similarity score: 0.7430034442672192


In [ ]:
import os
import tiktoken
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Input string
text = "Hello, I am learning how to tokenize and compute embeddings."

# --- Tokenization ---
encoding = tiktoken.encoding_for_model("gpt-4")
tokens = encoding.encode(text)

print("Tokens with text and first 5 embedding values:\n")

# --- Embeddings ---
embeddings_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=[encoding.decode([t]) for t in tokens]  # one embedding per token
)

# --- Print side by side ---
for token, emb in zip(tokens, embeddings_response.data):
    token_text = encoding.decode([token])
    embedding = emb.embedding
    print(f"{token:<8} | {token_text:<12} | {embedding[:5]}")


Tokens with text and first 5 embedding values:

9906     | Hello        | [0.01925872266292572, -0.06453847140073776, -0.0016806893981993198, 0.07808944582939148, 0.021657824516296387]
11       | ,            | [0.012168597429990768, -0.010514059104025364, 0.057705070823431015, 0.04873959347605705, 0.03059672750532627]
358      |  I           | [-0.017859503626823425, -0.004921667277812958, -0.01970040611922741, 0.014617317356169224, 0.014205174520611763]
1097     |  am          | [-0.006633950863033533, -0.019582634791731834, -0.013453628867864609, -0.005034953821450472, 0.032200489193201065]
6975     |  learning    | [-0.015062838792800903, -0.037228479981422424, -0.018344823271036148, 0.01976538449525833, 0.04734385013580322]
1268     |  how         | [-0.02521488443017006, -0.014974553138017654, -0.010812811553478241, 0.06329450756311417, -0.005326000973582268]
311      |  to          | [0.034410543739795685, -0.00937828328460455, -0.0003765611909329891, 0.044689830392599106, 0.010

In [ ]:
# To show activations as vectors during presentation

from transformers import AutoTokenizer, AutoModel
import torch

torch.set_printoptions(precision=15, sci_mode=False)

# Choose a small model for demo (faster to run in Colab)
model_name = "bert-base-uncased"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)

# Input text
text = "The cat sat on the mat."

# Tokenize and convert to tensor
inputs = tokenizer(text, return_tensors="pt")

# Forward pass (get outputs including hidden states)
with torch.no_grad():
    outputs = model(**inputs)

# Hidden states: tuple of (layer_0, layer_1, ..., layer_N)
hidden_states = outputs.hidden_states

print(f"Number of layers (including embedding layer): {len(hidden_states)}")
print(f"Shape of embeddings for each layer: {[h.shape for h in hidden_states]}")

# Example: print activations (vectors) from layer 1 for the first token
layer = 1
token_index = 0
print(f"\nActivation vector from layer {layer}, token {token_index}:")
print(hidden_states[layer][0][token_index])

Number of layers (including embedding layer): 13
Shape of embeddings for each layer: [torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768]), torch.Size([1, 9, 768])]

Activation vector from layer 1, token 0:
tensor([ 0.063050836324692,  0.077486075460911, -0.214047208428383,
        -0.297945022583008,  0.244390517473221,  0.061978198587894,
        -0.270316600799561, -0.042023770511150, -0.094746872782707,
        -0.434608966112137,  0.032188773155212, -0.049290653318167,
         0.191111326217651, -0.070772826671600, -0.017468530684710,
         0.504784107208252,  0.484503656625748,  0.387925982475281,
        -0.050608057528734, -0.721040189266205, -0.676887810230255,
        -0.359229803085327, -0.159978657960892, -0.376345992088318,
   

In [ ]:
# To show weights and bias as vectors during presentation

from transformers import AutoModel

# Load a small model
model_name = "bert-base-uncased"
model = AutoModel.from_pretrained(model_name)

# Example: Pick the first linear layer in BERT encoder
first_linear = model.encoder.layer[0].attention.output.dense

# Get weights and bias
weights = first_linear.weight   # shape: (output_dim, input_dim)
bias = first_linear.bias        # shape: (output_dim)

print("Weights shape:", weights.shape)
print("Bias shape:", bias.shape)

# Convert to numpy for easier handling
import numpy as np
w = weights.detach().cpu().numpy()
b = bias.detach().cpu().numpy()

# Print a sample
print("\nFirst 25 weights of first row as vector:\n", w[0][:25])
print("\nFirst 25 bias values:\n", b[:25])


Weights shape: torch.Size([768, 768])
Bias shape: torch.Size([768])

First 25 weights of first row as vector:
 [ 0.0058194916 -0.0029993376  0.0028637904  0.011584928  -0.00232744
  0.029671706   0.010161855  -0.016970493   0.0007698858  0.023937644
 -0.02442923   -0.018711017   0.024155473  -0.000219215   0.010842978
  0.009846834   0.014087142  -0.0026688972 -0.025023742   0.0018954846
 -0.0066417484  0.060299683   0.035193626  -0.016249916   0.004076755 ]

First 25 bias values:
 [ 0.005110629  -0.016662499   0.028129384  -0.011660613   0.019426273
 -0.0431647    -0.016972234   0.00857614   -0.013620353   0.013163552
  0.025706118   0.052135155   0.05140883   -0.031390827  -0.017878389
  0.031021204   0.03179858   -0.01129934   -0.0024910546 -0.0023036369
 -0.000527399   0.017469827  -0.016861966   0.018516552   0.002411837 ]


In [ ]:
!pip install -U weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 599.9/599.9 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.3 MB/s eta 0:00:00


In [ ]:
# To connect to Weaviate cloud vector database

import os
import weaviate
from weaviate.classes.init import Auth

# Best practice: store your credentials in environment variables
weaviate_url = os.environ["WEAVIATE_URL"]
weaviate_api_key = os.environ["WEAVIATE_API_KEY"]

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=Auth.api_key(weaviate_api_key),
)

print(client.is_ready())

True


In [ ]:
# To create collection in weaviate cloud vector database

from weaviate.classes.config import Configure

client.collections.create(
    "VisCollection",
    vector_config=[
        Configure.Vectors.text2vec_weaviate(
            name="title_vector",
            source_properties=["title"],
            model="Snowflake/snowflake-arctic-embed-l-v2.0"
        )
    ],
    # Additional parameters not shown
)

In [ ]:
# To store data in Weaviate cloud vector database

source_objects = [
    {"title": "The Shawshank Redemption", "description": "A wrongfully imprisoned man forms an inspiring friendship while finding hope and redemption in the darkest of places."},
    {"title": "The Godfather", "description": "A powerful mafia family struggles to balance loyalty, power, and betrayal in this iconic crime saga."},
    {"title": "The Dark Knight", "description": "Batman faces his greatest challenge as he battles the chaos unleashed by the Joker in Gotham City."},
    {"title": "Jingle All the Way", "description": "A desperate father goes to hilarious lengths to secure the season's hottest toy for his son on Christmas Eve."},
    {"title": "A Christmas Carol", "description": "A miserly old man is transformed after being visited by three ghosts on Christmas Eve in this timeless tale of redemption."}
]

collection = client.collections.use("VisCollection")

with collection.batch.fixed_size(batch_size=200) as batch:
    for src_obj in source_objects:
        # The model provider integration will automatically vectorize the object
        batch.add_object(
            properties={
                "title": src_obj["title"],
                "description": src_obj["description"],
            },
            # vector=vector  # Optionally provide a pre-obtained vector
        )
        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.")
            break

failed_objects = collection.batch.failed_objects
if failed_objects:
    print(f"Number of failed imports: {len(failed_objects)}")
    print(f"First failed object: {failed_objects[0]}")

In [ ]:
# To query for data in Weaviate cloud vector database

collection = client.collections.use("VisCollection")

response = collection.query.near_text(
    query="A holiday film",  # The model provider integration will automatically vectorize the query
    limit=2
)

for obj in response.objects:
    print(obj.properties["title"])

A Christmas Carol
The Shawshank Redemption


In [ ]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.9/745.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


In [ ]:
# connect to pinecone cloud
from pinecone import Pinecone
pc = Pinecone(api_key="redacted")

In [ ]:

pc.describe_index(name="vis-pinecone-index")

{
    "name": "vis-pinecone-index",
    "metric": "cosine",
    "host": "vis-pinecone-index-nz8fdqs.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1024,
    "deletion_protection": "disabled",
    "tags": null,
    "embed": {
        "model": "llama-text-embed-v2",
        "field_map": {
            "text": "text"
        },
        "dimension": 1024,
        "metric": "cosine",
        "write_parameters": {
            "dimension": 1024,
            "input_type": "passage",
            "truncate": "END"
        },
      

In [ ]:
# upsert data into pinecode

# To get the unique host for an index,
# see https://docs.pinecone.io/guides/manage-data/target-an-index
index = pc.Index(host="vis-pinecone-index-nz8fdqs.svc.aped-4627-b74a.pinecone.io")

# Upsert records into a vis-pinecone-namespace
data = [
    {"id": "vec1", "text": "Apple is a popular fruit known for its sweetness and crisp texture."},
    {"id": "vec2", "text": "The tech company Apple is known for its innovative products like the iPhone."},
    {"id": "vec3", "text": "Many people enjoy eating apples as a healthy snack."},
    {"id": "vec4", "text": "Apple Inc. has revolutionized the tech industry with its sleek designs and user-friendly interfaces."},
    {"id": "vec5", "text": "An apple a day keeps the doctor away, as the saying goes."},
    {"id": "vec6", "text": "Apple Computer Company was founded on April 1, 1976, by Steve Jobs, Steve Wozniak, and Ronald Wayne as a partnership."}
]

index.upsert_records(
    namespace="vis-pinecone-namespace",
    records=data
)

UpsertResponse(upserted_count=6, _response_info={'raw_headers': {'date': 'Sun, 07 Dec 2025 21:22:01 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '1276', 'server': 'envoy'}})

In [ ]:
# Querey the pinecone vector database
query = "Tell me about the tech company known as Apple."

results = index.search(
    namespace="vis-pinecone-namespace",
    query={
        "inputs": {"text": query},
        "top_k": 3
    }
)

print(results)

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '555',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 07 Dec 2025 21:22:03 GMT',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '149',
                                    'x-pinecone-api-version': '2025-10',
                                    'x-pinecone-max-indexed-lsn': '1'}},
 'result': {'hits': [{'_id': 'vec2',
                      '_score': 0.6304798722267151,
                      'fields': {'text': 'The tech company Apple is known for '
                                         'its innovative products like the '
                                         'iPhone.'}},
                     {'_id': 'vec4',
                      '_score': 0.536880612373352,
                      'fields': {'text': 'Apple Inc. has revo